# 00 · Setup

1. Checks the four conda envs and the GPU.
2. Logs in to Hugging Face (needed to publish, and for gated models if any).
3. Shows the config and your links file.

Edit `configs/config.yaml` (thresholds, ASR backend, prompts, publishing) and `links.txt` (playlists / videos / channels, one per line) before running the pipeline.

In [ ]:
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO)); os.chdir(REPO)
if Path("/workspace").exists():
    os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
import numpy as np, pandas as pd
import plotly.express as px
from egyspeech import nb
cfg, lay = nb.cfg_and_layout()          # configs/config.yaml
print("work dir:", lay.root)

In [ ]:
import subprocess
for env in (cfg.envs.main, cfg.envs.nemo, cfg.envs.qwen, "egyspeech-vllm"):
    py = Path(cfg.envs.conda_dir) / "envs" / env / "bin" / "python"
    if not py.exists():
        print(f"{env:18s} not installed"); continue
    out = subprocess.run([str(py), "-c", "import torch;print(torch.__version__, torch.cuda.is_available())"], capture_output=True, text=True)
    print(f"{env:18s} torch {out.stdout.strip() or out.stderr.strip()[-200:]}")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
from huggingface_hub import login
login()   # or set HF_TOKEN

In [ ]:
print(Path(cfg["_path"]).read_text(encoding="utf-8")[:4000])

In [ ]:
links = Path(cfg.links_file) if Path(cfg.links_file).is_absolute() else REPO / cfg.links_file
print(links, "exists:", links.exists())
if links.exists(): print(links.read_text(encoding="utf-8"))

In [ ]:
nb.run("status")